# Автоматическая адаптация англоязычных датасетов под культурные нормы России

**Пайплайн:**
1. Загрузка датасетов (CrowS-Pairs, SNIPS)
2. NER-разметка культурно-специфичных сущностей
3. Метод 1: Наивный машинный перевод (Google Translate)
4. Метод 2: Наивный zero-shot LLM перевод
5. Метод 3: Полный пайплайн адаптации (NER → перевод сущностей → несколько вариантов → LLM-Judge → лучший)
6. Сохранение всех вариантов датасетов
7. Оценка: Bias-метрика (CrowS-Pairs), Slot F1 + Intent Accuracy (SNIPS)
8. Проверка культурного сдвига через LLM
9. Агрегированное сравнение результатов

In [ ]:
%cd /kaggle/working
!git clone https://github.com/karlstedt020/adaptation.git

In [ ]:
%cd /kaggle/working/adaptation

In [ ]:
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, "D:/adaptation")

import logging
import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import PipelineConfig
from src.data_loader import load_crows_pairs, load_snips, snips_to_bio
from src.ner_processor import NERProcessor, entities_from_dicts
from src.mapping_store import MappingStore
from src.llm_client import LLMClient
from src.rate_limiter import parallel_apply, RateLimiter
from src.checkpoint import Checkpointer
from src.naive_translator import (
    translate_crows_pairs as naive_translate_crows,
    translate_snips as naive_translate_snips,
)
from src.naive_llm_translator import (
    translate_crows_pairs as llm_translate_crows,
    translate_snips as llm_translate_snips,
)
from src.pipeline_adapter import adapt_crows_pairs, adapt_snips
from src.judge import JudgeEvaluator
from src.label_checker import check_crows_pair, check_snips_utterance
from src.bias_evaluator import run_bias_evaluation
from src.slot_evaluator import run_slot_evaluation
from src.cultural_shift_checker import run_shift_evaluation
from src.visualization import (
    plot_bias_comparison, plot_slot_comparison,
    plot_cultural_shift, build_summary_table, format_summary,
)

print("All modules imported successfully.")

## 1. Конфигурация

Укажите ваш API-ключ OpenRouter и модель.

In [ ]:
cfg = PipelineConfig()
cfg.paths.ensure_dirs()

# ===== ЗАПОЛНИТЕ ВАШИ ДАННЫЕ =====
cfg.openrouter.api_key = ""   # <-- ваш ключ OpenRouter

# По умолчанию установлены дешёвые модели:
#   NER:       google/gemini-2.0-flash-001  (~$0.10 / $0.40 за 1M токенов)
#   Основная:  deepseek/deepseek-chat-v3.1  (~$0.14 / $0.28 за 1M токенов)
# Можно переопределить при необходимости:
# cfg.openrouter.model     = "anthropic/claude-sonnet-4.5"
# cfg.openrouter.ner_model = "openai/gpt-4o-mini"

# ── Ablation-флаги (экономия токенов) ─────────────────────────────
# use_rag=False    — не впрыскивать накопленные маппинги в system-prompt
# use_judge=False  — не делать отдельный LLM-вызов на выбор лучшего
# n_variants=1     — автоматически отключает judge (всё равно выбор единственный)
# Для максимальной экономии: n_variants=1, use_rag=False, use_judge=False
cfg.openrouter.n_variants = 3
cfg.openrouter.use_rag    = True
cfg.openrouter.use_judge  = True

# Сколько строк брать на smoke-check перед прогоном на всём датасете
SMOKE_LIMIT_CROWS = 5   # CrowS-Pairs: 5 пар хватит, чтобы увидеть работу
SMOKE_LIMIT_SNIPS = 5   # SNIPS: 5 высказываний
# =================================

print(f"Data root: {cfg.paths.root}")
print(f"Main model: {cfg.openrouter.model or 'NOT SET'}")
print(f"NER model:  {cfg.openrouter.ner_model or 'NOT SET'}")
print(f"n_variants={cfg.openrouter.n_variants} | "
      f"use_rag={cfg.openrouter.use_rag} | use_judge={cfg.openrouter.use_judge}")
print(f"Bias eval models: {cfg.evaluation.bias_models}")
print(f"Slot eval models: {cfg.evaluation.slot_models}")
print(f"Runs per evaluation: {cfg.evaluation.n_runs}")

## 2. Загрузка датасетов

In [ ]:
# Загрузка CrowS-Pairs
crows_df = load_crows_pairs(cfg.paths)
print(f"CrowS-Pairs: {len(crows_df)} pairs")
print(f"Bias types: {crows_df['bias_type'].value_counts().to_dict()}")
crows_df.head(3)

In [ ]:
# Загрузка SNIPS + BIO-теги (NER запустим в §3 после инициализации ner)
snips_df = load_snips(cfg.paths)
snips_df["bio_tags"] = snips_df.apply(lambda r: snips_to_bio(r.to_dict()), axis=1)
print(f"SNIPS: {len(snips_df)} utterances, {snips_df['intent'].nunique()} intents")
snips_df[["text", "intent"]].head(3)

## 3. NER-разметка культурно-специфичных сущностей

In [ ]:
# Инициализация LLM-клиентов: основного и отдельного для NER
assert cfg.openrouter.api_key, "Установите cfg.openrouter.api_key выше!"

llm     = LLMClient(cfg.openrouter)
ner_llm = LLMClient(cfg.openrouter, model_override=cfg.openrouter.ner_model)
ner     = NERProcessor(ner_llm)

print(f"Main LLM: {cfg.openrouter.model}")
print(f"NER LLM:  {cfg.openrouter.ner_model}")
print(f"Workers:  {cfg.openrouter.max_workers}  |  LLM RPS: {cfg.openrouter.rps}"
      f"  |  Translator RPS: {cfg.openrouter.translator_rps}")

# ── Тест NER ─────────────────────────────────────────────────
for test_sent in [
    "My mom spent all day cooking for Thanksgiving",
    crows_df.iloc[0]["sent_more"],
]:
    ents = ner.extract_cultural_entities(test_sent)
    print(f"\n'{test_sent}'")
    for e in ents:
        print(f"  [{e.label}] '{e.text}'  ({e.adaptation_type})")

print(f"\nNER LLM cache size after test: {ner_llm.cache_size}")

In [ ]:
# ── Параллельный NER на всём CrowS-Pairs ──────────────────────────────────
# Результаты сохраняются в df-колонках ner_more / ner_less / ner_diff_str.
# Pipeline будет читать их оттуда, не делая повторных API-вызовов.

ner_ck_crows = Checkpointer(cfg.paths.data_raw / "crows_ner.csv", ["ner_more"])

done_ids, existing_ner = ner_ck_crows.load()
pending_ner = crows_df[~crows_df.index.isin(done_ids)].copy()

if len(pending_ner):
    pending_ner["ner_more"] = parallel_apply(
        lambda t: json.dumps([e.to_dict() for e in ner.extract_cultural_entities(t)], ensure_ascii=False),
        pending_ner["sent_more"].tolist(),
        max_workers=cfg.openrouter.max_workers, rps=cfg.openrouter.rps,
        desc="NER CrowS sent_more",
    )
    pending_ner["ner_less"] = parallel_apply(
        lambda t: json.dumps([e.to_dict() for e in ner.extract_cultural_entities(t)], ensure_ascii=False),
        pending_ner["sent_less"].tolist(),
        max_workers=cfg.openrouter.max_workers, rps=cfg.openrouter.rps,
        desc="NER CrowS sent_less",
    )

crows_df = ner_ck_crows.merge_and_save(existing_ner, pending_ner) if len(pending_ner) else existing_ner

# Десериализуем обратно для статистики
crows_df["ner_more_parsed"] = crows_df["ner_more"].apply(
    lambda s: json.loads(s) if isinstance(s, str) else (s or [])
)
crows_df["ner_less_parsed"] = crows_df["ner_less"].apply(
    lambda s: json.loads(s) if isinstance(s, str) else (s or [])
)
crows_df["has_cultural_ents"] = crows_df["ner_more_parsed"].apply(bool)

n_with = crows_df["has_cultural_ents"].sum()
print(f"CrowS-Pairs с культурными сущностями: {n_with}/{len(crows_df)} ({n_with/len(crows_df)*100:.1f}%)")
print(f"NER LLM cache size: {ner_llm.cache_size}")

# Топ-10 самых частых сущностей
from collections import Counter
all_ents = [e["text"] for row in crows_df["ner_more_parsed"] for e in row]
print("\nТоп-10 сущностей:")
for ent, cnt in Counter(all_ents).most_common(10):
    print(f"  {ent!r}: {cnt}")

In [ ]:
# ── Параллельный NER на SNIPS ─────────────────────────────────────────────
ner_ck_snips = Checkpointer(cfg.paths.data_raw / "snips_ner.csv", ["ner_entities"])
done_ids_s, existing_ner_s = ner_ck_snips.load()
pending_ner_s = snips_df[~snips_df.index.isin(done_ids_s)].copy()

if len(pending_ner_s):
    pending_ner_s["ner_entities"] = parallel_apply(
        lambda t: json.dumps([e.to_dict() for e in ner.extract_cultural_entities(t)], ensure_ascii=False),
        pending_ner_s["text"].tolist(),
        max_workers=cfg.openrouter.max_workers, rps=cfg.openrouter.rps,
        desc="NER SNIPS",
    )

snips_df = ner_ck_snips.merge_and_save(existing_ner_s, pending_ner_s) if len(pending_ner_s) else existing_ner_s
snips_df["ner_entities_parsed"] = snips_df["ner_entities"].apply(
    lambda s: json.loads(s) if isinstance(s, str) else (s or [])
)
snips_df["has_cultural_ents"] = snips_df["ner_entities_parsed"].apply(bool)

print(f"SNIPS с культурными сущностями: {snips_df['has_cultural_ents'].sum()}/{len(snips_df)}")
print(f"NER LLM cache size: {ner_llm.cache_size}")
snips_df[["text", "intent", "ner_entities_parsed"]].head(3)

## 4. Метод 1: Наивный машинный перевод (Google Translate)

Простой перевод без культурной адаптации — сохраняет американские реалии на русском языке.

### 4.0 Smoke-check: наивный перевод на первых N записях

Перед полным прогоном убедимся, что перевод работает и имеет смысл.

In [ ]:
# Smoke-check наивного MT на первых N записях
_smoke_crows_naive = naive_translate_crows(
    crows_df, limit=SMOKE_LIMIT_CROWS,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.translator_rps,
)
print("=== CrowS-Pairs: Naive MT ===")
for _, r in _smoke_crows_naive.iterrows():
    print(f"[{r['bias_type']}]")
    print(f"  EN more: {r['sent_more']}")
    print(f"  RU more: {r['sent_more_ru']}")
    print(f"  EN less: {r['sent_less']}")
    print(f"  RU less: {r['sent_less_ru']}")
    print()

_smoke_snips_naive = naive_translate_snips(
    snips_df, limit=SMOKE_LIMIT_SNIPS,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.translator_rps,
)
print("=== SNIPS: Naive MT ===")
for _, r in _smoke_snips_naive.iterrows():
    print(f"[{r['intent']}]  EN: {r['text']}\n                RU: {r['text_ru']}\n")

In [ ]:
# CrowS-Pairs: наивный параллельный MT с checkpoint
crows_naive = naive_translate_crows(
    crows_df,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.translator_rps,
    checkpoint=Checkpointer(
        cfg.paths.data_naive_translated / "crows_pairs_naive.csv",
        result_cols=["sent_more_ru", "sent_less_ru"],
    ),
)
crows_naive.to_csv(cfg.paths.data_naive_translated / "crows_pairs_naive.csv", index=True)
print(f"Сохранено: {cfg.paths.data_naive_translated / 'crows_pairs_naive.csv'}")
crows_naive[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru"]].head(3)

In [ ]:
# SNIPS: наивный параллельный MT с checkpoint
snips_naive = naive_translate_snips(
    snips_df,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.translator_rps,
    checkpoint=Checkpointer(
        cfg.paths.data_naive_translated / "snips_naive.jsonl",
        result_cols=["text_ru"],
    ),
)
snips_naive.to_json(cfg.paths.data_naive_translated / "snips_naive.jsonl",
                   orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_naive_translated / 'snips_naive.jsonl'}")
snips_naive[["text", "text_ru", "intent"]].head(3)

## 5. Метод 2: Наивный zero-shot LLM перевод

LLM переводит с минимальным промптом — без структурированной методологии адаптации.

In [ ]:
# LLM-клиент уже создан выше (в ячейке с инициализацией NER).
print(f"Используем основную LLM: {cfg.openrouter.model}")

### 5.0 Smoke-check: zero-shot LLM перевод на первых N записях

In [ ]:
# Smoke-check zero-shot LLM перевода
_smoke_crows_llm = llm_translate_crows(
    llm, crows_df, limit=SMOKE_LIMIT_CROWS,
    max_workers=cfg.openrouter.max_workers, rps=cfg.openrouter.rps,
)
print("=== CrowS-Pairs: Zero-shot LLM ===")
for _, r in _smoke_crows_llm.iterrows():
    print(f"[{r['bias_type']}]")
    print(f"  EN more: {r['sent_more']}")
    print(f"  RU more: {r['sent_more_ru']}")
    print(f"  EN less: {r['sent_less']}")
    print(f"  RU less: {r['sent_less_ru']}")
    print()

_smoke_snips_llm = llm_translate_snips(
    llm, snips_df, limit=SMOKE_LIMIT_SNIPS,
    max_workers=cfg.openrouter.max_workers, rps=cfg.openrouter.rps,
)
print("=== SNIPS: Zero-shot LLM ===")
for _, r in _smoke_snips_llm.iterrows():
    print(f"[{r['intent']}]  EN: {r['text']}\n                RU: {r['text_ru']}\n")

empty_crows = (_smoke_crows_llm["sent_more_ru"] == "").sum()
empty_snips = (_smoke_snips_llm["text_ru"] == "").sum()
print(f"Empty: CrowS={empty_crows}/{len(_smoke_crows_llm)}, SNIPS={empty_snips}/{len(_smoke_snips_llm)}")
assert empty_crows == 0 and empty_snips == 0, "⚠ Есть пустые переводы — проверьте модель/промпт"

In [ ]:
# CrowS-Pairs: zero-shot LLM параллельный перевод с checkpoint
crows_llm = llm_translate_crows(
    llm, crows_df,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
    checkpoint=Checkpointer(
        cfg.paths.data_naive_llm / "crows_pairs_llm.csv",
        result_cols=["sent_more_ru", "sent_less_ru"],
    ),
)
crows_llm.to_csv(cfg.paths.data_naive_llm / "crows_pairs_llm.csv", index=True)
print(f"Сохранено: {cfg.paths.data_naive_llm / 'crows_pairs_llm.csv'}")
print(f"LLM cache size: {llm.cache_size}")
crows_llm[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru"]].head(3)

In [ ]:
# SNIPS: zero-shot LLM параллельный перевод с checkpoint
snips_llm = llm_translate_snips(
    llm, snips_df,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
    checkpoint=Checkpointer(
        cfg.paths.data_naive_llm / "snips_llm.jsonl",
        result_cols=["text_ru"],
    ),
)
snips_llm.to_json(cfg.paths.data_naive_llm / "snips_llm.jsonl",
                  orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_naive_llm / 'snips_llm.jsonl'}")
snips_llm[["text", "text_ru", "intent"]].head(3)

## 6. Метод 3: Полный пайплайн культурной адаптации

**Шаги:**
1. NER-детекция культурно-специфичных сущностей
2. Генерация N вариантов адаптации через LLM с Anthropological Prompting
3. LLM-as-a-Judge выбирает лучший вариант
4. Проверка сохранения лейбла
5. Обновление базы маппингов для консистентности

In [ ]:
# Инициализация компонентов пайплайна
mapping_store = MappingStore(cfg.paths.data_adapted / "mappings.json")
judge = JudgeEvaluator(llm)

print(f"Mapping store: {len(mapping_store)} existing mappings")
print(f"Variants per example: {cfg.openrouter.n_variants}")

### 6.0 Smoke-check: полный пайплайн на первых N записях

In [ ]:
# Smoke-check полного пайплайна (отдельный маппинг-стор, не мешает финальному)
_smoke_mapping = MappingStore(cfg.paths.data_adapted / "mappings_smoke.json")
_smoke_judge = JudgeEvaluator(llm)

_smoke_crows_pipeline = adapt_crows_pairs(
    llm, ner, _smoke_mapping, _smoke_judge, crows_df,
    n_variants=cfg.openrouter.n_variants,
    limit=SMOKE_LIMIT_CROWS,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
)
print("=== CrowS-Pairs: Pipeline ===")
for _, r in _smoke_crows_pipeline.iterrows():
    print(f"[{r['bias_type']}]")
    print(f"  EN more: {r['sent_more']}")
    print(f"  RU more: {r['sent_more_ru']}")
    print(f"  EN less: {r['sent_less']}")
    print(f"  RU less: {r['sent_less_ru']}")
    print()

_smoke_snips_pipeline = adapt_snips(
    llm, _smoke_mapping, _smoke_judge, snips_df,
    n_variants=cfg.openrouter.n_variants,
    limit=SMOKE_LIMIT_SNIPS,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
)
print("=== SNIPS: Pipeline ===")
for _, r in _smoke_snips_pipeline.iterrows():
    print(f"[{r['intent']}]  EN: {r['text']}\n           RU: {r['text_ru']}\n  slots: {r['slots_ru']}\n")

empty_c = (_smoke_crows_pipeline["sent_more_ru"] == "").sum()
empty_s = (_smoke_snips_pipeline["text_ru"] == "").sum()
assert empty_c == 0, f"⚠ {empty_c} пустых пар CrowS pipeline"
assert empty_s == 0, f"⚠ {empty_s} пустых SNIPS pipeline"
print(f"✓ Smoke-check пройден. Маппингов собрано: {len(_smoke_mapping)}")
print("Можно запускать полный прогон ниже.")

In [ ]:
# CrowS-Pairs: полный пайплайн — параллельно, с checkpoint, NER из df
mapping_store = MappingStore(cfg.paths.data_adapted / "mappings.json")
judge = JudgeEvaluator(llm)

crows_adapted = adapt_crows_pairs(
    llm, ner, mapping_store, judge, crows_df,
    n_variants=cfg.openrouter.n_variants,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
    use_rag=cfg.openrouter.use_rag,
    use_judge=cfg.openrouter.use_judge,
    checkpoint=Checkpointer(
        cfg.paths.data_adapted / "crows_pairs_adapted.csv",
        result_cols=["sent_more_ru", "sent_less_ru"],
    ),
)
crows_adapted.to_csv(cfg.paths.data_adapted / "crows_pairs_adapted.csv", index=True)
print(f"Сохранено: {cfg.paths.data_adapted / 'crows_pairs_adapted.csv'}")
print(f"Маппингов в базе: {len(mapping_store)}  |  LLM cache: {llm.cache_size}")
crows_adapted[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru", "bias_type"]].head(3)

In [ ]:
# SNIPS: полный пайплайн — параллельно, с checkpoint, NER из df
snips_adapted = adapt_snips(
    llm, mapping_store, judge, snips_df,
    n_variants=cfg.openrouter.n_variants,
    max_workers=cfg.openrouter.max_workers,
    rps=cfg.openrouter.rps,
    use_rag=cfg.openrouter.use_rag,
    use_judge=cfg.openrouter.use_judge,
    checkpoint=Checkpointer(
        cfg.paths.data_adapted / "snips_adapted.jsonl",
        result_cols=["text_ru"],
    ),
)
snips_adapted.to_json(cfg.paths.data_adapted / "snips_adapted.jsonl",
                      orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_adapted / 'snips_adapted.jsonl'}")
print(f"Маппингов в базе: {len(mapping_store)}  |  LLM cache: {llm.cache_size}")
snips_adapted[["text", "text_ru", "intent"]].head(3)

In [ ]:
# Просмотр базы маппингов
import json
mappings = json.loads((cfg.paths.data_adapted / "mappings.json").read_text(encoding="utf-8"))
print(f"Всего маппингов: {len(mappings)}")
# Показать первые 20
for orig, adapted in list(mappings.items())[:20]:
    print(f"  {orig} → {adapted}")

## 7. Проверка сохранения лейблов (выборочная)

Используем LLM для cross-check — корректны ли лейблы после адаптации.

In [ ]:
# Выборочная проверка лейблов CrowS-Pairs (на 30 примерах)
label_check_sample = crows_adapted.sample(n=min(30, len(crows_adapted)), random_state=42)
valid_count = 0
issues = []

for _, row in label_check_sample.iterrows():
    try:
        result = check_crows_pair(
            llm,
            original={"sent_more": row["sent_more"], "sent_less": row["sent_less"],
                       "bias_type": row["bias_type"], "stereo_antistereo": row["stereo_antistereo"]},
            adapted={"sent_more_ru": row["sent_more_ru"], "sent_less_ru": row["sent_less_ru"]},
        )
        if result.get("valid"):
            valid_count += 1
        else:
            issues.append({"index": row.name, "issues": result.get("issues", [])})
    except Exception as e:
        issues.append({"index": row.name, "issues": [str(e)]})

print(f"Валидных пар: {valid_count}/{len(label_check_sample)} ({valid_count/len(label_check_sample)*100:.0f}%)")
if issues:
    print(f"\nПримеры проблем:")
    for iss in issues[:5]:
        print(f"  idx={iss['index']}: {iss['issues']}")

## 8. Оценка: CrowS-Pairs Bias Metric

Pseudo-log-likelihood metric: % пар, где модель предпочитает стереотипное предложение.
Прогоняем на нескольких MLM-моделях × всех методах перевода.

In [ ]:
import gc, torch

# Оценка bias на всех вариантах датасета
crows_datasets = {
    "Naive MT": crows_naive,
    "Zero-shot LLM": crows_llm,
    "Pipeline": crows_adapted,
}

bias_results = {}
for method, df in crows_datasets.items():
    print(f"\n{'='*50}")
    print(f"Evaluating bias: {method}")
    print(f"{'='*50}")
    bias_results[method] = run_bias_evaluation(
        df,
        model_names=cfg.evaluation.bias_models,
        sent_more_col="sent_more_ru",
        sent_less_col="sent_less_ru",
        n_runs=cfg.evaluation.n_runs,
    )
    print(bias_results[method].groupby("model")["metric_score"].agg(["mean", "std"]))

    # Чистим память после каждого метода (run_bias_evaluation уже чистит после
    # каждой модели внутри, но здесь гарантируем финальную очистку)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"GPU memory after {method}: "
              f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## 9. Оценка: SNIPS Slot Filling & Intent Classification

Дообучаем BERT-модели на каждом варианте перевода SNIPS, тестируем на одном тест-сете.

> **Примечание**: для корректной оценки slot filling нужны BIO-теги для переведённых текстов. Для наивного перевода теги берутся из оригинала (позиционное приближение), для pipeline-адаптации — из ответа LLM.

In [ ]:
# Подготовка BIO-тегов для переведённых версий SNIPS
# Для наивного перевода: токены могут не совпадать, используем оригинальные теги с обрезкой
def align_bio_tags_naive(row):
    """Грубое выравнивание BIO-тегов: берём оригинальные, обрезаем/дополняем по длине."""
    if not row.get("text_ru") or pd.isna(row["text_ru"]):
        return ["O"]
    n_tokens_ru = len(str(row["text_ru"]).split())
    orig_tags = row["bio_tags"]
    if n_tokens_ru <= len(orig_tags):
        return orig_tags[:n_tokens_ru]
    return orig_tags + ["O"] * (n_tokens_ru - len(orig_tags))

# Применяем к наивному и LLM-переводу
for df_name, df in [("snips_naive", snips_naive), ("snips_llm", snips_llm)]:
    df["bio_tags_ru"] = df.apply(lambda r: align_bio_tags_naive(r.to_dict()), axis=1)
    print(f"{df_name}: bio_tags_ru added")

# Для pipeline-адаптации: используем slots_ru из ответа LLM
def build_bio_from_slots_ru(row):
    """Построить BIO-теги из slots_ru для адаптированного текста."""
    text = row.get("text_ru", "")
    slots = row.get("slots_ru", [])
    if not text or pd.isna(text) or not slots:
        return ["O"] * max(len(str(text).split()), 1)
    tokens = str(text).split()
    tags = ["O"] * len(tokens)
    for slot in slots:
        if not isinstance(slot, dict):
            continue
        slot_text = slot.get("text", "")
        slot_name = slot.get("slot", "UNK")
        # Простой поиск подстроки
        slot_tokens = slot_text.split()
        for i in range(len(tokens) - len(slot_tokens) + 1):
            if tokens[i:i+len(slot_tokens)] == slot_tokens:
                tags[i] = f"B-{slot_name}"
                for j in range(1, len(slot_tokens)):
                    tags[i+j] = f"I-{slot_name}"
                break
    return tags

snips_adapted["bio_tags_ru"] = snips_adapted.apply(
    lambda r: build_bio_from_slots_ru(r.to_dict()), axis=1
)
print("snips_adapted: bio_tags_ru added")

In [ ]:
import gc, torch
from sklearn.model_selection import train_test_split

# Train/test split (80/20) на ИНДЕКСАХ snips_df — один и тот же сплит для всех методов.
# Важно: используем df.index.isin(...), а не iloc с позиционными индексами,
# иначе при фильтрации пустых переводов строки «съезжают».
train_pos, test_pos = train_test_split(
    range(len(snips_df)), test_size=0.2,
    random_state=42, stratify=snips_df["intent"],
)
train_ids = set(snips_df.index[list(train_pos)])
test_ids  = set(snips_df.index[list(test_pos)])

snips_methods = {
    "Naive MT": snips_naive,
    "Zero-shot LLM": snips_llm,
    "Pipeline": snips_adapted,
}

slot_results = {}
for method, df in snips_methods.items():
    print(f"\n{'='*50}\nEvaluating slots: {method}\n{'='*50}")
    valid_mask = df["text_ru"].notna() & (df["text_ru"] != "")
    train_df = df[df.index.isin(train_ids) & valid_mask].copy()
    test_df  = df[df.index.isin(test_ids)  & valid_mask].copy()
    print(f"  train={len(train_df)}  test={len(test_df)}")

    slot_results[method] = run_slot_evaluation(
        train_df, test_df,
        model_names=cfg.evaluation.slot_models,
        text_col="text_ru",
        bio_tags_col="bio_tags_ru",
        n_runs=cfg.evaluation.n_runs,
        epochs=cfg.evaluation.slot_epochs,
        batch_size=cfg.evaluation.slot_batch_size,
        lr=cfg.evaluation.slot_lr,
    )
    print(slot_results[method].groupby("model")[["intent_accuracy", "slot_f1"]].agg(["mean", "std"]))

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"GPU memory after {method}: "
              f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## 10. Проверка культурного сдвига через LLM

LLM оценивает качество культурной адаптации (1-5) на выборке из каждого метода.

In [ ]:
# Проверка культурного сдвига на CrowS-Pairs
shift_results = run_shift_evaluation(
    llm,
    datasets=crows_datasets,
    sent_col="sent_more_ru",
    sample_size=100,
    n_runs=cfg.evaluation.n_runs,
)

print("Результаты проверки культурного сдвига:")
print(shift_results.groupby("method")[["mean_score"]].agg(["mean", "std"]))

## 11. Агрегированные результаты и визуализация

Сводная таблица и графики по всем метрикам × всем методам × всем моделям.

In [ ]:
# Сводная таблица
summary = build_summary_table(bias_results, slot_results, shift_results)
formatted = format_summary(summary)
print("=" * 80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (mean ± std по нескольким прогонам)")
print("=" * 80)
formatted

In [ ]:
# График: Bias metric по методам и моделям
fig_bias = plot_bias_comparison(bias_results)
fig_bias.savefig(cfg.paths.root / "bias_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# График: Slot filling / Intent classification по методам и моделям
fig_slot = plot_slot_comparison(slot_results)
fig_slot.savefig(cfg.paths.root / "slot_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# График: Cultural shift quality
fig_shift = plot_cultural_shift(shift_results)
fig_shift.savefig(cfg.paths.root / "cultural_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Детальная таблица bias по типам bias
print("BIAS METRIC ПО ТИПАМ ПРЕДУБЕЖДЕНИЙ")
print("=" * 80)
for method, df in bias_results.items():
    print(f"\n--- {method} ---")
    bias_cols = [c for c in df.columns if c.startswith("bias_")]
    if bias_cols:
        print(df[["model"] + bias_cols].groupby("model").mean().round(3).to_string())

## 12. Сохранение всех результатов

In [ ]:
# Сохранение результатов оценки
results_dir = cfg.paths.root / "results"
results_dir.mkdir(exist_ok=True)

# Bias results
for method, df in bias_results.items():
    df.to_csv(results_dir / f"bias_{method.replace(' ', '_').lower()}.csv", index=False)

# Slot results
for method, df in slot_results.items():
    df.to_csv(results_dir / f"slot_{method.replace(' ', '_').lower()}.csv", index=False)

# Shift results
shift_results.to_csv(results_dir / "cultural_shift.csv", index=False)

# Summary
summary.to_csv(results_dir / "summary.csv", index=False)
formatted.to_csv(results_dir / "summary_formatted.csv", index=False)

print(f"Все результаты сохранены в {results_dir}")
print(f"\nФайлы:")
for f in sorted(results_dir.glob("*.csv")):
    print(f"  {f.name}")

## 13. Ключевое сравнение: оригинал (EN) vs адаптированный pipeline (RU)

Для CrowS-Pairs и SNIPS прогоняем **одни и те же мультиязычные модели** на двух версиях данных:

| Датасет | Источник | Что мерим |
|---|---|---|
| CrowS-Pairs | оригинал EN (`sent_more` / `sent_less`) | bias-metric **без** дообучения |
| CrowS-Pairs | pipeline RU (`sent_more_ru` / `sent_less_ru`) | bias-metric **без** дообучения |
| SNIPS | оригинал EN (`text` / `bio_tags`) | обучаем BERT и меряем intent-acc + slot-F1 на EN-test |
| SNIPS | pipeline RU (`text_ru` / `bio_tags_ru`) | обучаем BERT и меряем intent-acc + slot-F1 на RU-test |

Цель — показать, насколько меняется качество, когда модель тренируется на культурно-адаптированном датасете вместо западного оригинала.

In [ ]:
# ── 13.1 CrowS-Pairs: bias на оригинале EN vs pipeline RU ───────────
# Одни и те же мультиязычные BERT'ы, два варианта данных. Без дообучения.
import gc, torch

bias_crosslang = {}

print("=== CrowS EN (оригинал) ===")
bias_crosslang["Original EN"] = run_bias_evaluation(
    crows_df,
    model_names=cfg.evaluation.bias_models_cross,
    sent_more_col="sent_more",
    sent_less_col="sent_less",
    n_runs=cfg.evaluation.n_runs,
)
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n=== CrowS RU (pipeline) ===")
bias_crosslang["Pipeline RU"] = run_bias_evaluation(
    crows_adapted,
    model_names=cfg.evaluation.bias_models_cross,
    sent_more_col="sent_more_ru",
    sent_less_col="sent_less_ru",
    n_runs=cfg.evaluation.n_runs,
)
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

for name, df in bias_crosslang.items():
    print(f"\n{name}:")
    print(df.groupby("model")["metric_score"].agg(["mean", "std"]).round(3))

In [ ]:
# ── 13.2 Диаграмма: CrowS bias — EN vs RU ─────────────────────
fig_bias_xl = plot_bias_comparison(
    bias_crosslang,
    title="CrowS-Pairs Bias: Original EN vs Pipeline RU",
)
fig_bias_xl.savefig(cfg.paths.root / "bias_en_vs_ru.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 13.3 SNIPS: обучение на EN-train → тест на EN-test, vs обучение на RU-train → тест на RU-test
# Для корректного сравнения используем один и тот же сплит индексов snips_df.
# Для EN: text / bio_tags (они есть в snips_df).
# Для RU: text_ru / bio_tags_ru (из snips_adapted).
import gc, torch

slot_crosslang = {}

# EN: обучаем и тестируем на оригинале
print("=== SNIPS EN (оригинал) ===")
train_en = snips_df[snips_df.index.isin(train_ids)].copy()
test_en  = snips_df[snips_df.index.isin(test_ids)].copy()
print(f"  train={len(train_en)}  test={len(test_en)}")

slot_crosslang["Original EN"] = run_slot_evaluation(
    train_en, test_en,
    model_names=cfg.evaluation.slot_models_cross,
    text_col="text",
    bio_tags_col="bio_tags",
    n_runs=cfg.evaluation.n_runs,
    epochs=cfg.evaluation.slot_epochs,
    batch_size=cfg.evaluation.slot_batch_size,
    lr=cfg.evaluation.slot_lr,
)
print(slot_crosslang["Original EN"].groupby("model")[["intent_accuracy", "slot_f1"]].agg(["mean", "std"]).round(3))
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

# RU: обучаем и тестируем на pipeline-адаптированной версии
print("\n=== SNIPS RU (pipeline) ===")
valid_mask_ru = snips_adapted["text_ru"].notna() & (snips_adapted["text_ru"] != "")
train_ru = snips_adapted[snips_adapted.index.isin(train_ids) & valid_mask_ru].copy()
test_ru  = snips_adapted[snips_adapted.index.isin(test_ids)  & valid_mask_ru].copy()
print(f"  train={len(train_ru)}  test={len(test_ru)}")

slot_crosslang["Pipeline RU"] = run_slot_evaluation(
    train_ru, test_ru,
    model_names=cfg.evaluation.slot_models_cross,
    text_col="text_ru",
    bio_tags_col="bio_tags_ru",
    n_runs=cfg.evaluation.n_runs,
    epochs=cfg.evaluation.slot_epochs,
    batch_size=cfg.evaluation.slot_batch_size,
    lr=cfg.evaluation.slot_lr,
)
print(slot_crosslang["Pipeline RU"].groupby("model")[["intent_accuracy", "slot_f1"]].agg(["mean", "std"]).round(3))
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── 13.4 Диаграмма: SNIPS intent-acc & slot-F1 — EN vs RU ──────
fig_slot_xl = plot_slot_comparison(
    slot_crosslang,
    title="SNIPS: Train/Test on Original EN vs Pipeline RU",
)
fig_slot_xl.savefig(cfg.paths.root / "slot_en_vs_ru.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 13.5 Сохранение сырых результатов cross-lingual сравнения ──
xl_dir = cfg.paths.root / "results"
xl_dir.mkdir(exist_ok=True)
for name, df in bias_crosslang.items():
    df.to_csv(xl_dir / f"bias_xlang_{name.replace(' ', '_').lower()}.csv", index=False)
for name, df in slot_crosslang.items():
    df.to_csv(xl_dir / f"slot_xlang_{name.replace(' ', '_').lower()}.csv", index=False)
print(f"EN-vs-RU результаты сохранены в {xl_dir}")

## Сохранённые датасеты

| Метод | CrowS-Pairs | SNIPS |
|-------|-------------|-------|
| Наивный MT | `data/naive_translated/crows_pairs_naive.csv` | `data/naive_translated/snips_naive.jsonl` |
| Zero-shot LLM | `data/naive_llm/crows_pairs_llm.csv` | `data/naive_llm/snips_llm.jsonl` |
| Pipeline | `data/adapted/crows_pairs_adapted.csv` | `data/adapted/snips_adapted.jsonl` |
| Маппинги | — | `data/adapted/mappings.json` |